Historical raw-data cleaning and EDA. Preserved for provenance. Some imputation uses other columns including price; do not use these full-data operations to claim leakage-free evaluation. The supplied cleaned snapshot is the reproducible input to training/train.py. Regenerated files go to training/runs/cleaning, leaving supplied datasets intact.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# For preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor

# For data transformation
from scipy.stats import boxcox


In [ ]:

# Load the data
print("Loading data...")
df = pd.read_csv(ROOT / "data/raw/fusion_2.csv")

# Make a copy to avoid modifying the original data
data = df.copy()
print("nombre des lignes est\n" , data.shape[0],"Nombre des colonnes ",data.shape[1])
print(data.head()) 
print(data.info())
print("nomrbre des doublons",data.duplicated().sum())

# Display initial missing values
print("Missing values before cleaning:\n", data.isnull().sum())



In [ ]:

# 1. INITIAL DATA CLEANING
print("\n1. INITIAL DATA CLEANING")

# Drop Type column as it's not needed for prediction
data['Type'].unique()

data.drop('Type', axis=1, inplace=True)
# drop duplicate
data.drop_duplicates(inplace=True)  
# Rename columns to remove spaces

data.columns = data.columns.str.replace(" ", "")


In [ ]:

# 2. CLEANING KILOMETRAGE
print("\n2. CLEANING KILOMETRAGE")

# Clean Kilométrage column
data['Kilométrage'] = data['Kilométrage'].str.replace(' ', '', regex=True)
data['Kilométrage'] = data['Kilométrage'].replace('Plusde500000', '500000-549999')

# Split range values and calculate average
data[['Km_1', 'Km_2']] = data['Kilométrage'].str.split('-', expand=True)
data['Km_1'] = pd.to_numeric(data['Km_1'], errors='coerce')
data['Km_2'] = pd.to_numeric(data['Km_2'], errors='coerce')
data['Kilométrage'] = (data['Km_1'] + data['Km_2']) / 2
data.drop(['Km_1', 'Km_2'], axis=1, inplace=True)
# presque toutes les features de ces lignes sont null
df_test=data[data['Kilométrage'].isnull()] 
print(df_test.head())
print(df_test.isnull().sum())
print(df_test.shape)
# Drop rows with missing Kilométrage as it's a crucial feature
data.dropna(subset=['Kilométrage'], inplace=True)


In [ ]:
data.dropna(subset=['Kilométrage'], inplace=True)
# Replace NaN values in column with the mode
mode_year = data['Model_Year'].mode()[0]
data['Model_Year'] = data['Model_Year'].fillna(mode_year)
data['Model_Year'] = data['Model_Year'].astype(int)
data.dropna(subset=['Carburant'], inplace=True)
data.dropna(subset=['Puissance'], inplace=True)
data['Puissance'] = data['Puissance'].str.extract(r'(\d+)').astype(float)

print("Valeurs monquantes par colonne : \n", data.isnull().sum())
data.shape


In [ ]:
df_test=data[data[['Nb_portes', 'Origine', 'Première_main', 'État']].isnull().all(axis=1)] #[1041 rows avec 4 Colonnes manquantes
print(df_test)        
df_test.isnull().sum()
data.dropna(subset=['Nb_portes', 'Origine', 'Première_main', 'État'], how='all', inplace=True)

# 1. Standardisation des portes à 2 ou 4
data['Nb_portes'] = data['Nb_portes'].replace({5: 4, 3: 2})

# 2. Préparer les données d’entraînement (sans NaN)
df_KNN = data[data['Nb_portes'].notna()].copy()
X_train = pd.get_dummies(df_KNN[['Marque', 'Model']], drop_first=True)
y_train = df_KNN['Nb_portes'].astype(int)
# 4. entraînement du modèle final avec best_k
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
# 5. Imputation des valeurs manquantes
df_pred = data[data['Nb_portes'].isna()].copy()
X_test = pd.get_dummies(df_pred[['Marque', 'Model']], drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

data.loc[data['Nb_portes'].isna(), 'Nb_portes'] = knn.predict(X_test)
data['Nb_portes'] = data['Nb_portes'].astype(int)


In [ ]:

# 4. CLEAN CATEGORICAL VARIABLES
# Drop rows with missing Carburant (fuel type)
# List all categorical columns to be cleaned
categorical_cols = [
    'Première_main', 'État', 'Origine',
    'Transmission', 'Carburant', 'Marque', 'Model'
]

# Normalize text (lowercase + strip whitespace)
for col in categorical_cols:
    if col in data.columns:
        data[col] = data[col].astype(str).str.lower().str.strip()
        data[col] = data[col].replace('nan', np.nan)
# Standardize État (condition) values

data['État'] = data['État'].replace({
'endommagé': 'mauvais',
'pour pièces': 'mauvais',
'correct': 'correct',
'bon': 'bon',
'très bon': 'très bon',
'excellent': 'excellent',
'neuf': 'neuf'
    })



In [ ]:
# 5. MISSING VALUES IMPUTATION - ADVANCED (Anass)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd

def impute_lr(df, target, *, features=None, encode=False, max_iter=1000):
    """Fill NaNs in *target* with a Logistic‑Regression prediction."""
    known, unknown = df[df[target].notna()], df[df[target].isna()]
    if unknown.empty:               # rien à prédire
        return

    if encode:                      # one‑hot + réalignement
        Xk = pd.get_dummies(known.drop(columns=target), drop_first=True)
        Xu = (pd.get_dummies(unknown.drop(columns=target), drop_first=True)
                 .reindex(columns=Xk.columns, fill_value=0))
    else:                           # colonnes numériques déjà prêtes
        Xk, Xu = known[features], unknown[features]

    scaler = StandardScaler().fit(Xk)
    Xk, Xu = scaler.transform(Xk), scaler.transform(Xu)

    preds = LogisticRegression(max_iter=max_iter).fit(Xk, known[target]).predict(Xu)
    df.loc[unknown.index, target] = preds


num_feats = [c for c in ['Model_Year', 'Kilométrage', 'Nb_portes', 'Puissance'] if c in data.columns]

# ── imputations ─────────────────────────────────────────────────────────────────
impute_lr(data, 'Première_main', encode=True)                 # 1ʳᵉ main
impute_lr(data, 'État',    features=num_feats, max_iter=2000) # État
impute_lr(data, 'Origine', features=num_feats, max_iter=2000) # Origine

print("Missing values after imputation:\n", data.isnull().sum())
data.shape


In [ ]:
# 6. OUTLIER DETECTION AND HANDLING
print("\n6. OUTLIER DETECTION AND HANDLING")

# Function to display outlier stats for numerical columns
def show_outlier_stats(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    n_outliers_low = (df[col] < lower_bound).sum()
    n_outliers_high = (df[col] > upper_bound).sum()
    
    print(f"{col}:")
    print(f"  Lower bound: {lower_bound:.2f}, Upper bound: {upper_bound:.2f}")
    print(f"  Outliers: {n_outliers_low + n_outliers_high} ({n_outliers_low} below, {n_outliers_high} above)")
    print(f"  Percentage: {((n_outliers_low + n_outliers_high) / len(df)) * 100:.2f}%")
    
    return lower_bound, upper_bound

# Visualize numerical columns distribution
numerical_cols = ['Price', 'Kilométrage', 'Model_Year']

if 'Puissance' in data.columns:
    numerical_cols.append('Puissance')

plt.figure(figsize=(14, 10))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(len(numerical_cols), 2, i*2-1)
    sns.boxplot(x=data[col])
    plt.title(f'Boxplot: {col}')
    
    plt.subplot(len(numerical_cols), 2, i*2)
    sns.histplot(data[col], kde=True)
    plt.title(f'Distribution: {col}')

plt.tight_layout()
plt.savefig('outliers_before.png')
plt.show()
plt.close()


In [ ]:
numerical_cols = ['Kilométrage', 'Model_Year', 'Price']  # Ajoute d'autres si besoin

# Liste des marques de luxe qu'on garde
marques_de_luxe = ['lexus', 'tesla', 'mercedes-benz', 'porsche', 'bmw', 'audi', 'range rover', 'jaguar', 'bentley', 'rolls-royce']

# Supprimer les valeurs absurdes
data = data[~((data['Price'] > 800000) & (~data['Marque'].isin(marques_de_luxe)))]
data = data[data['Price'] < 2000000]
data = data[data['Price'] > 10000]

def remove_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = max(Q1 - 1.5 * IQR,0)
    upper_bound = Q3 + 1.5 * IQR
    
    print(f"{col} : Suppression des valeurs hors [{lower_bound:.2f}, {upper_bound:.2f}]")
    
    # Filtrer les données
    return df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

# Appliquer à chaque colonne numérique
for col in numerical_cols:
    data = remove_outliers_iqr(data, col)


In [ ]:
# 1. Suppression des valeurs dupliquees :
print("Le nombre de ligne avant la suppression est : ", data.shape[0])
print("nomrbre des doublons",data.duplicated().sum())
data = data.drop_duplicates()
print("Le nombre de ligne apres la suppression est : ", data.shape[0])
print("nomrbre des doublons",data.duplicated().sum())


In [ ]:
data_preprocessed = data.copy()


In [ ]:
# ---------- 7. FEATURE ENGINEERING + LOG‑PRICE ------------------
import numpy as np
import pandas as pd

def add_features(df: pd.DataFrame, *, reference_year=None) -> pd.DataFrame:
    """Return a copy of *df* with engineered predictors (no leakage)."""
    out = df.copy()

    if reference_year is None:
        reference_year = out['Model_Year'].max()
    out['Car_Age'] = reference_year - out['Model_Year']

    # 4. Log‑price (target)
    out['Log_Price'] = np.log1p(out['Price'])

    return out
# ----------------------------------------------------------------
# Apply feature engineering
data = add_features(data)


In [ ]:

# Préparer les données
values = data['Price'].values

# Tracé
plt.figure(figsize=(10, 4))
plt.hist(values, bins=30, density=True)               # histogramme normalisé
mu, sigma = values.mean(), values.std()               # paramètres de la normale
x = np.linspace(mu - 3*sigma, mu + 3*sigma, 100)
plt.plot(x, stats.norm.pdf(x, mu, sigma),             # densité normale
linestyle='--', linewidth=2, label='Loi Normale')
# Finitions
plt.title("Histogramme de la variable Price vs loi normale")
plt.legend()
plt.show()
plt.figure(figsize=(10, 6))
stats.probplot(data['Price'], dist="norm", plot=plt)
plt.title("Q-Q Plot du price")
plt.show()
skewness = data['Price'].skew()
kurtosis = data['Price'].kurt()

print(f"Asymétrie du price avant normalisation (skewness) = {skewness:.2f}")
print(f"Aplatissement du price avant normalisation (kurtosis) = {kurtosis:.2f}")

data['Log_Price'] = np.log1p(data['Price'])
skewness = data['Log_Price'] .skew()
kurtosis = data['Log_Price'] .kurt()

print(f"Asymétrie apres normalisation(skewness) = {skewness:.2f}")
print(f"Aplatissement apres normalisation(kurtosis) = {kurtosis:.2f}")

stats.probplot( data['Log_Price'], dist="norm", plot=plt)
plt.title("apres la transformation")
plt.tight_layout() #pour eviter le chauvechement des graphes 
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(data['Kilométrage'], kde=True, stat='density', color='black', bins=30)

# Superposer une courbe normale théorique
mu, sigma = data['Kilométrage'].mean(), data['Kilométrage'].std()
x = np.linspace(mu - 3*sigma, mu + 3*sigma, 100)
plt.plot(x, stats.norm.pdf(x, mu, sigma), 'r--', linewidth=2, label='Distribution Normale')

plt.title("Histogramme du Kilométrage vs Loi Normale")
plt.legend()
plt.show()
from sklearn.mixture import GaussianMixture
import numpy as np
X = data[['Kilométrage']].dropna().values
bic_scores = []

for k in range(1, 6):
    gmm = GaussianMixture(n_components=k).fit(X)
    bic_scores.append(gmm.bic(X))

plt.plot(range(1, 6), bic_scores, marker='o')
plt.title("GMM BIC Scores pour détecter le nombre de groupes")
plt.xlabel("Nombre de composantes")
plt.ylabel("BIC")
plt.show()
skewness = data['Kilométrage'] .skew()
kurtosis = data['Kilométrage'] .kurt()
print(f"Asymétrie de kilométrage(skewness) = {skewness:.2f}")
print(f"Aplatissement de kilométrage(kurtosis) = {kurtosis:.2f}")


In [ ]:
print(data.columns)


In [ ]:
# 8. ENCODING AND FINAL PREPROCESSING
print("\n8. ENCODING AND FINAL PREPROCESSING")

# Define encoding for different column types
binary_cols = ['Première_main', 'Transmission']  # Binary encoding
ordinal_cols = ['État']  # Ordinal encoding 
ohe_cols = ['Carburant', 'Origine']  # One-hot encoding
freq_cols = ['Marque', 'Model']  # Frequency encoding
passthrough_cols = ['Nb_portes']
# Binary encoding
for col in binary_cols:
    if col in data.columns:
        mapping = {'oui': 1, 'non': 0, 'automatique': 1, 'manuelle': 0}
        data[col] = data[col].map(mapping)

# Ordinal encoding for État (condition)
if 'État' in data.columns:
    etat_order = {
        'mauvais': 0,
        'correct': 1,
        'bon': 2,
        'très bon': 3,
        'excellent': 4,
        'neuf': 5
    }
    data['État'] = data['État'].map(etat_order)

# One-hot encoding
for col in ohe_cols:
    if col in data.columns:
        dummies = pd.get_dummies(data[col], prefix=col, drop_first=True)
        data = pd.concat([data, dummies], axis=1)
        data.drop(col, axis=1, inplace=True)

# Frequency encoding
for col in freq_cols:
    if col in data.columns:
        freq_map = data[col].value_counts(normalize=True)
        data[col + '_freq'] = data[col].map(freq_map)
        data.drop(col, axis=1, inplace=True)

# Standardize numerical features
numerical_cols = ['Kilométrage', 'Car_Age']

if 'Puissance' in data.columns:
    numerical_cols.append('Puissance')

scaler = StandardScaler()
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

print("Standardized numerical features", numerical_cols)

print("Final dataframe shape:", data.shape)



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Calcul de la matrice de corrélation
corr_matrix = data.corr()
target='Price'
# 2. Extraction des corrélations avec la cible
corr_with_target = corr_matrix[target].sort_values(ascending=False)

# 3. Conversion en DataFrame et transposition pour affichage horizontal
corr_df = corr_with_target.to_frame(name='corr_with_target').T
top_corr = corr_with_target.head(10).to_frame(name='corr_with_target').T

# 4. Création de la figure avec deux sous-graphiques côte à côte
plt.figure(figsize=(12, 4))

# Heatmap complète
plt.subplot(2, 1, 1)
sns.heatmap(
    corr_df,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    center=0
)
plt.title("Corrélation des features avec la cible")
plt.xlabel("Features")
plt.ylabel("Corrélation")

# Heatmap des top 10 features
plt.subplot(2, 1, 2)
sns.heatmap(
    top_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    center=0
)
plt.title("Corrélation des top 10 features avec la cible")
plt.xlabel("Top 10 Features")
plt.ylabel("Corrélation")

plt.subplots_adjust(wspace=0.4, hspace=8)


plt.show()


In [ ]:
# 1. Chargement et préparation

corr_matrix = data.corr()

# 2. Tracé de la heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.4,
    cbar_kws={"shrink": 0.7}
)
plt.title("Heatmap de corrélation des variables")
plt.tight_layout()
plt.show()


In [ ]:
# 1. Calcul de la matrice de corrélation
corr_matrix = data.corr().abs()  # .abs() pour avoir la valeur absolue des corrélations

# 2. Masquer la partie supérieure de la matrice (triangulaire supérieure)
mask = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
strong_corrs = corr_matrix.where(mask)

# 3. Trouver les paires dont la corrélation est > 0.7
threshold = 0.7
strong_pairs = strong_corrs.stack().reset_index()
strong_pairs.columns = ['Variable 1', 'Variable 2', 'Corrélation']
strong_pairs = strong_pairs[strong_pairs['Corrélation'] > threshold]

# 4. Affichage
print("Paires de variables fortement corrélées :")
print(strong_pairs)


In [ ]:
data.drop(columns=['Model_Year','Log_Price'],axis=1,inplace=True)
# data_preprocessed.drop(columns=['Model_Year'],axis=1,inplace=True)
print(data.columns)


In [ ]:
print(data_preprocessed.columns)


In [ ]:
import pandas as pd
from pathlib import Path

# your final DataFrame is called `data`
out_file = ROOT / "training/runs/cleaning/cars_encoded.csv"
out_file.parent.mkdir(parents=True, exist_ok=True)   # create folders if needed

data.to_csv(out_file, index=False)
print(f"✅ Saved: {out_file.resolve()}")


In [ ]:
# your final DataFrame is called `data`
out_file_2 = ROOT / "training/runs/cleaning/cars_preprocessed.csv"
out_file_2.parent.mkdir(parents=True, exist_ok=True)   # create folders if needed

data_preprocessed.to_csv(out_file_2, index=False)
print(f"✅ Saved: {out_file_2.resolve()}")
